# WTA Doubles Player Analysis

This notebook fetches WTA doubles player data from the Tennis API and displays it using pandas DataFrames.

## 1. Import Required Libraries

In [1]:
%pip install --break-system-packages requests pandas python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install pandas python-dotenv
import os
import requests
import pandas as pd
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Set Up API Request

Store your RapidAPI key in a **`.env`** file in this folder (same directory as the notebook). Copy **`.env.example`** to **`.env`**, then set `RAPIDAPI_KEY=...`. Never commit `.env`. The next cell reads the key with `os.getenv("RAPIDAPI_KEY")` after `load_dotenv()` runs in the imports cell.

In [3]:
# API configuration — key from .env (RAPIDAPI_KEY) or your shell environment
url = "https://tennis-api-atp-wta-itf.p.rapidapi.com/tennis/v2/atp/fixtures/2024-02-07"

api_key = os.getenv("RAPIDAPI_KEY")
if not api_key:
    raise ValueError(
        "RAPIDAPI_KEY is not set. Copy `.env.example` to `.env` in this folder and add your RapidAPI key."
    )

headers = {
    "x-rapidapi-key": api_key,
    "x-rapidapi-host": "tennis-api-atp-wta-itf.p.rapidapi.com",
    "Content-Type": "application/json",
}

print(f"API URL: {url}")
print("RapidAPI key loaded from environment (key value is not printed).")

API URL: https://tennis-api-atp-wta-itf.p.rapidapi.com/tennis/v2/atp/fixtures/2024-02-07
RapidAPI key loaded from environment (key value is not printed).


## 3. Load WTA Doubles Data into DataFrame

In [4]:
# Create sample WTA doubles player data since the API returned empty data
wta_doubles_data = {
    'Player 1': ['Barbora Krejcikova', 'Ekaterina Makarova', 'Rajeev Ram', 'Wesley Koolhof', 'Jean-Julien Rojer'],
    'Player 2': ['Katerina Siniakova', 'Elena Vesnina', 'Francoise Abanda', 'Nikola Mektic', 'Marcelo Arevalo'],
    'Country 1': ['Czech Republic', 'Russia', 'Canada', 'Netherlands', 'Netherlands'],
    'Country 2': ['Czech Republic', 'Russia', 'Canada', 'Croatia', 'El Salvador'],
    'Ranking': [1, 5, 12, 3, 8],
    'Tournament': ['Australian Open', 'Miami Open', 'Roland Garros', 'Wimbledon', 'US Open']
}

# Create DataFrame from the sample data
df_wta_doubles = pd.DataFrame(wta_doubles_data)

# Display the first 5 rows
print("First 5 rows of WTA Doubles Players:")
print(df_wta_doubles.head())

First 5 rows of WTA Doubles Players:
             Player 1            Player 2       Country 1       Country 2  \
0  Barbora Krejcikova  Katerina Siniakova  Czech Republic  Czech Republic   
1  Ekaterina Makarova       Elena Vesnina          Russia          Russia   
2          Rajeev Ram    Francoise Abanda          Canada          Canada   
3      Wesley Koolhof       Nikola Mektic     Netherlands         Croatia   
4   Jean-Julien Rojer     Marcelo Arevalo     Netherlands     El Salvador   

   Ranking       Tournament  
0        1  Australian Open  
1        5       Miami Open  
2       12    Roland Garros  
3        3        Wimbledon  
4        8          US Open  


In [5]:
# Display dataframe information
print("DataFrame Information:")
df_wta_doubles.info()

DataFrame Information:
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Player 1    5 non-null      str  
 1   Player 2    5 non-null      str  
 2   Country 1   5 non-null      str  
 3   Country 2   5 non-null      str  
 4   Ranking     5 non-null      int64
 5   Tournament  5 non-null      str  
dtypes: int64(1), str(5)
memory usage: 372.0 bytes


## 4. Check DataFrame Structure

In [20]:
# Count WTA doubles players by country
# Combine both Country 1 and Country 2 to get total count per country
country_1_counts = df_wta_doubles['Country 1'].value_counts()
country_2_counts = df_wta_doubles['Country 2'].value_counts()

# Combine and sum the counts
all_countries = pd.concat([country_1_counts, country_2_counts]).groupby(level=0).sum().sort_values(ascending=False)

print("WTA Doubles Players Count by Country:")
print(all_countries)

WTA Doubles Players Count by Country:
Canada            2
Czech Republic    2
Netherlands       2
Russia            2
Croatia           1
El Salvador       1
Name: count, dtype: int64


## 5. Top 30 WTA doubles players by country

Each player and their country is listed on one row so we can filter with `df[df['Country'] == some_country]`. The **30 “top”** players are the 30 best (lowest) `Ranking` values in that country.

In [6]:
# One row per (Player, Country); same team row yields two player rows, same Ranking on both
p1 = df_wta_doubles[['Player 1', 'Country 1', 'Ranking']].rename(
    columns={'Player 1': 'Player', 'Country 1': 'Country'}
)
p2 = df_wta_doubles[['Player 2', 'Country 2', 'Ranking']].rename(
    columns={'Player 2': 'Player', 'Country 2': 'Country'}
)
df = pd.concat([p1, p2], ignore_index=True).drop_duplicates().sort_values('Ranking')

for country in sorted(df['Country'].unique()):
    # Boolean indexing: df[condition] keeps rows where Country matches
    top30 = df[df['Country'] == country].sort_values('Ranking').head(30)
    print(f"\n--- Top 30 in {country} (by best available ranking) ---")
    print(top30.to_string(index=False))


--- Top 30 in Canada (by best available ranking) ---
          Player Country  Ranking
      Rajeev Ram  Canada       12
Francoise Abanda  Canada       12

--- Top 30 in Croatia (by best available ranking) ---
       Player Country  Ranking
Nikola Mektic Croatia        3

--- Top 30 in Czech Republic (by best available ranking) ---
            Player        Country  Ranking
Barbora Krejcikova Czech Republic        1
Katerina Siniakova Czech Republic        1

--- Top 30 in El Salvador (by best available ranking) ---
         Player     Country  Ranking
Marcelo Arevalo El Salvador        8

--- Top 30 in Netherlands (by best available ranking) ---
           Player     Country  Ranking
   Wesley Koolhof Netherlands        3
Jean-Julien Rojer Netherlands        8

--- Top 30 in Russia (by best available ranking) ---
            Player Country  Ranking
Ekaterina Makarova  Russia        5
     Elena Vesnina  Russia        5


## 6. Group player data by country and ranking

`df.groupby(['Country', 'Ranking'])` puts every player row into a bucket defined by both **country** and **ranking** (same country + same ranking value together). You can then count rows, list names, or aggregate per group.

In [1]:
# df comes from the previous cell (one row per Player, Country, Ranking)
by_country_rank = df.groupby(['Country', 'Ranking'], sort=True)

print("How many players in each (Country, Ranking) group:")
print(by_country_rank.size().to_string())

print("\nPlayer names in each group:")
print(by_country_rank['Player'].agg(lambda s: ', '.join(s)).to_string())

How many players in each (Country, Ranking) group:
Country         Ranking
Canada          12         2
Croatia         3          1
Czech Republic  1          2
El Salvador     8          1
Netherlands     3          1
                8          1
Russia          5          2

Player names in each group:
Country         Ranking
Canada          12                   Rajeev Ram, Francoise Abanda
Croatia         3                                   Nikola Mektic
Czech Republic  1          Barbora Krejcikova, Katerina Siniakova
El Salvador     8                                 Marcelo Arevalo
Netherlands     3                                  Wesley Koolhof
                8                               Jean-Julien Rojer
Russia          5               Ekaterina Makarova, Elena Vesnina


## 7. API request

In [ ]:
# Make API request
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise exception for bad status codes
    data = response.json()
    print(f"API request successful!")
    print(f"Response status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching data: {e}")
    data = None

API request successful!
Response status code: 200


## 8. Count missing values

`df.isnull()` marks each cell as True if it is missing (`NaN` / `NA`). Calling `.sum()` on that counts how many missing values exist **in each column** (because True counts as 1).

In [1]:
# Uses the same df as sections 5–6 (run those cells first so df exists)
print("Missing cells per column:")
print(df.isnull().sum())

print("\nTotal missing cells in the table:", int(df.isnull().sum().sum()))

Missing cells per column:
Player     0
Country    0
Ranking    0
dtype: int64

Total missing cells in the table: 0
